# 01：多轮对话与工作记忆增长
原理：Checkpointer 保存会话状态，但“保存了什么”和“本次发给模型什么”是两个问题。
本实验用固定模型观察真实 `create_agent` 管线；固定模型只回显输入，不模拟语言理解。

在本章 `notebooks/` 目录运行。共用组件见 [context_lab.py](./context_lab.py)。
每个 Notebook 从头执行即可，不需要先执行其他 Notebook。真实模型单元默认关闭。


In [ ]:
import os
import json
from pathlib import Path
from uuid import uuid4
from context_lab import (
    initialize, DemoModel, live_model, dumps, estimate_messages, request_cost,
    WorkingState, Context, archive, read_archive, summary_middleware,
    demo_summary, valid_tool_pairs, recent_messages, SUMMARY_PROMPT,
)
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

DATA = initialize()
# 默认不调用外部模型。可在启动内核前设置 CH09_RUN_LIVE=1，或在此改为 True。
RUN_LIVE = os.getenv('CH09_RUN_LIVE', '0') == '1'
print('真实模型演示：', RUN_LIVE)


## 1. 捕获模型实际输入
`WorkingState.task` 保存当前有效任务，覆盖更新；`messages` 使用框架 reducer 追加。
本例的任务变更由应用显式传入，展示状态语义，并不声称模型已经自动抽取需求。


In [ ]:
seen = []
@wrap_model_call
def observe(request, handler):
    task = request.state.get('task', {})
    system = SystemMessage('当前有效任务：' + dumps(task))
    sent = [system, *request.messages]
    seen.append({'messages': [m.model_copy(deep=True) for m in sent],
                 'count': len(request.messages), 'estimate': request_cost(sent, request.tools)})
    return handler(request.override(system_message=system))

agent = create_agent(DemoModel(), state_schema=WorkingState, middleware=[observe],
                     checkpointer=InMemorySaver())
ctx = Context('alice', 'login-lab')
thread_id = 'growth-' + uuid4().hex
config = {'configurable': {'thread_id': thread_id}}
turns = [
    ('帮我设计登录功能，需求编号 REQ-009。', {'goal': '登录', 'id': 'REQ-009', 'methods': ['待确定']}),
    ('增加手机号验证码登录。', {'goal': '登录', 'id': 'REQ-009', 'methods': ['手机号验证码', '密码']}),
    ('密码登录暂缓，发送频率尚未确认。', {'goal': '登录', 'id': 'REQ-009', 'methods': ['手机号验证码'],
                                  'deferred': ['密码登录'], 'todo': ['发送频率尚未确认']}),
]
for text, task in turns:
    incoming = HumanMessage(text, id=uuid4().hex)
    archive(DATA, ctx, thread_id, [incoming])
    result = agent.invoke({'messages': [incoming], 'task': task}, config, context=ctx)
    archive(DATA, ctx, thread_id, result['messages'])
    print('State 消息数：', len(result['messages']), '输入估计：', seen[-1]['estimate'])
print('最近一次实际输入：')
for message in seen[-1]['messages']:
    print(message.type, str(message.content)[:220])


## 2. 验证会话继续、状态更正与新会话隔离
检查 State 与捕获的输入，不只看模型口头回答。原始消息另存 JSONL，后续压缩不影响来源回查。


In [ ]:
assert [x['count'] for x in seen] == [1, 3, 5]
assert len(result['messages']) == 6
assert result['task']['methods'] == ['手机号验证码']
assert result['task']['todo'] == ['发送频率尚未确认']
assert len(read_archive(DATA, ctx, thread_id)) == 6
old = agent.get_state(config).values
fresh = {'configurable': {'thread_id': 'fresh-' + uuid4().hex}}
other = agent.invoke({'messages': [HumanMessage('新会话开始')], 'task': {}}, fresh, context=ctx)
assert len(other['messages']) == 2
assert agent.get_state(config).values == old
assert [seen[i]['estimate'] for i in range(3)] == sorted(seen[i]['estimate'] for i in range(3))
print('通过：会话累积、当前状态更正、thread 隔离、原文回查。')


## 3. 可选真实模型
一次调用提供完整历史与当前状态，观察模型是否遵守“密码登录暂缓”。
断言检查有效状态与输入；回答语义由打印结果人工复核，不使用固定句子匹配。


In [ ]:
if RUN_LIVE:
    real = create_agent(live_model(), state_schema=WorkingState, middleware=[observe])
    # 使用明确标注的对话素材，避免把离线固定模型的回显文本当成真实助手历史。
    fixture = []
    for text, _ in turns:
        fixture += [HumanMessage(text), AIMessage('已记录本轮要求，后续更正优先。')]
    answer = real.invoke({'messages': fixture + [HumanMessage('列出当前有效需求和下一步。')],
                          'task': result['task']}, context=ctx)
    assert answer['task']['methods'] == ['手机号验证码']
    assert answer['messages'][-1].content
    print(answer['messages'][-1].content)
else:
    print('未执行真实模型调用。')


## 观察结论
消息持续增加；有效任务状态可以保持简洁。换 thread 不会自动继承工作记忆。
此时尚未实现情景召回和核心记忆，不能据此判断跨会话记忆失效。
